# scGPT — DIMER E2E cell-state classification tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/scgpt-single-cell-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/scgpt-single-cell-pipeline/blob/main/tutorials/scgpt_single_cell_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-tdc%2FscGPT-ffcc4d?style=flat)](https://huggingface.co/tdc/scGPT) [![Upstream](https://img.shields.io/badge/Upstream-bowang--lab%2FscGPT-181717?style=flat&logo=github&logoColor=white)](https://github.com/bowang-lab/scGPT) [![Paper](https://img.shields.io/badge/Nat%20Methods-10.1038%2Fs41592--024--02201--0-b31b1b.svg)](https://doi.org/10.1038/s41592-024-02201-0)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** single-cell expression embeddings, masked-expression prediction and bounded cell-state classification fine-tuning

**This notebook is standalone.** It carries the repository's package (3 modules under `src/scgpt_single_cell_pipeline/`, at revision `uncommitted`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `acf749f35bf5c0b00633838f02588272ed0d9911` (~205 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies (torch, safetensors, numpy, huggingface-hub — no transformers, no PyTDC), stages and digest-verifies the pinned scGPT snapshot (4 files, ~205 MB: config, weights and README from the Hub, the gene vocabulary from its persistent Harvard Dataverse file id), rebuilds the encoder on plain torch modules and loads the weights strictly, generates a deterministic 64-cell dataset in code from two real PBMC marker programmes (no download), validates the cells and the dataset contract, splits them into stratified train/validation/test sets, shows how a cell becomes (gene, bin) tokens, computes `<cls>` cell embeddings and checks they are reproducible and gene-order invariant, probes the masked-expression objective, measures a zero-training nearest-centroid baseline on the frozen embeddings plus majority-class and library-size baselines, runs a bounded AdamW fine-tuning of a classification head and the last encoder layer, evaluates accuracy, macro-F1 and AUROC on the held-out test split, classifies six freshly generated cells, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify prediction parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about a minute of model time after the download.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled cells as a genes-as-columns CSV (`id`, one column per gene symbol, `label`), a JSON array or a JSONL file of `{{id, counts, label}}` records. They pass through the same validation, stratified split, baselines, adaptation, held-out evaluation, inference, artifact export and reload-parity cells as the synthetic sample. The expected schema, the gene-symbol convention and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

scGPT is a single-cell foundation model: a 12-layer transformer encoder pretrained with a masked-expression objective on 33 million human cells (Cui et al., Nature Methods 2024). A cell enters the model as a set of **(gene token, expression bin)** pairs — every detected gene's expression is quantile-binned within the cell into 51 levels — with a `<cls>` token whose output is the cell embedding. There is no positional encoding, so the order in which genes are listed does not matter, which this notebook checks.

The checkpoint here is the *whole-human* encoder repackaged in safetensors by Therapeutics Data Commons. The Hub repository ships no model code, so this pipeline rebuilds the encoder on plain torch modules whose parameter names match the checkpoint exactly, and the gene vocabulary — which is not in the Hub repository — is fetched from its persistent Dataverse file id and digest-verified like everything else. Section 3 loads it; Section 7 probes what the pretrained decoder can and cannot do.

The tutorial dataset is synthetic but built from **real lineage programmes**: 24 canonical T-lymphocyte marker genes and 24 B-lymphocyte marker genes, plus 300 background genes from the vocabulary. A `t-like` cell places the T programme high and the B programme low; a `b-like` cell does the reverse; every cell is scaled to the same library size and expresses the same 348 genes, so total counts and gene detection carry no signal by construction.

**Learning objectives:** install the pinned runtime; inspect the carried pipeline, dataset and metrics modules; stage and digest-verify an immutable snapshot whose vocabulary comes from a second, non-Hub source; read how expression counts become (gene, bin) tokens and why ties are spread; validate cells and split a labelled dataset without leakage; extract `<cls>` cell embeddings and confirm they are reproducible and gene-order invariant; probe the masked-expression objective and read an honest negative result; measure a zero-training nearest-centroid baseline on the frozen embeddings and two trivial baselines; run a bounded fine-tuning with explicit hyperparameters; evaluate accuracy, macro-F1 and AUROC on an independent test split; classify new cells; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** multi-batch integration and batch-correction metrics, perturbation-response prediction, gene-network inference, the published scGPT benchmarks, the organ-specific scGPT checkpoints, reading AnnData `.h5ad` files directly, and any claim that a synthetic profile is a real cell. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). CPU is enough — a cell embeds in about 40 ms and the default fine-tuning takes about 20 s — and CUDA is used automatically when present.
- **Knowledge:** what a single-cell expression profile is, what library size and quantile binning mean, and how accuracy, macro-F1 and AUROC differ.
- **No remote code, no transformers:** the encoder is re-implemented on `torch.nn.TransformerEncoder` and three small blocks; the safetensors state dict loads with `strict=True`. Nothing from the Hub or Dataverse is executed.
- **Data contract:** records are `{{id, counts, label}}`, with `counts` a `{{gene symbol: count}}` mapping of non-negative numbers (raw or normalised — binning is rank-based within the cell); at least 10 detected genes per cell and at least 10 that resolve to a human gene symbol in the scGPT vocabulary (`GAPDH`, not `ENSG00000111640`); unique ids; at least 8 records and 3 per class, 2..20 classes. Cells with more than 1,535 detected genes keep the most expressed and report the truncation. BYOD accepts a genes-as-columns CSV, JSON array or JSONL; export an AnnData `.h5ad` to CSV first.
- **Validation is structural, not biological:** nothing checks that a profile is a real cell, that counts share a scale across cells, or that symbols follow one nomenclature.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — a patient-derived expression matrix is exactly that. The default path uploads nothing.
- **External access:** the Hugging Face Hub only, to fetch the pinned `tdc/scGPT` snapshot (~205 MB in total) at revision `acf749f35bf5…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `safetensors`, `numpy` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'huggingface-hub==1.32.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'scgpt-single-cell-pipeline',
    'repository_revision': 'uncommitted',
    'embedded_module': 'src/scgpt_single_cell_pipeline/pipeline.py',
    'embedded_modules': ['src/scgpt_single_cell_pipeline/metrics.py', 'src/scgpt_single_cell_pipeline/pipeline.py', 'src/scgpt_single_cell_pipeline/samples.py'],
    'module_sha256': '71a25ea948fbb0d6060eed4dc21d43e94b22b7120ce516bf0bf0c4bb95c9c270',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, safetensors, numpy
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'safetensors': safetensors.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/scgpt_single_cell_pipeline/` @ `uncommitted`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/scgpt_single_cell_pipeline/metrics.py`

In [ ]:
"""Classification metrics and trivial baselines for scGPT cell-state classification.

Pure Python (no scikit-learn): accuracy, macro-F1, per-class precision/recall/F1/support, and AUROC
(binary: positive class = the last entry of `classes`; multiclass: macro one-vs-rest), computed by
the Mann-Whitney rank statistic with average ranks for ties.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any


def _prf(hits: int, n_pred: int, n_true: int) -> dict[str, float]:
    precision = hits / n_pred if n_pred else 0.0
    recall = hits / n_true if n_true else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}


def auroc(y_true: Sequence[int], scores: Sequence[float]) -> float | None:
    """Area under the ROC curve for binary 0/1 labels; None when only one class is present."""
    n_pos = sum(1 for y in y_true if y == 1)
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    order = sorted(range(len(scores)), key=lambda i: scores[i])
    ranks = [0.0] * len(scores)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and scores[order[j + 1]] == scores[order[i]]:
            j += 1
        avg = (i + j + 2) / 2.0  # 1-based average rank of the tie block
        for k in range(i, j + 1):
            ranks[order[k]] = avg
        i = j + 1
    rank_sum = sum(r for r, y in zip(ranks, y_true, strict=True) if y == 1)
    return round((rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg), 4)


def classification_metrics(
    y_true: Sequence[str],
    y_pred: Sequence[str],
    scores: Sequence[Sequence[float]] | None,
    classes: Sequence[str],
) -> dict[str, Any]:
    """Discrete and ranking metrics over one evaluation split (labels are class names).

    `scores[i][k]` is the score of class `classes[k]` for record i (softmax outputs from the
    pipeline; any monotone score works for AUROC). Class order is preserved exactly as given.
    """
    if len(y_true) != len(y_pred):
        raise ValueError(f"{len(y_true)} labels vs {len(y_pred)} predictions")
    class_list = list(classes)
    unknown = sorted((set(y_true) | set(y_pred)) - set(class_list))
    if unknown:
        raise ValueError(f"labels outside the class list {class_list}: {unknown}")
    n = len(y_true)
    correct = sum(1 for t, p in zip(y_true, y_pred, strict=True) if t == p)
    per_class: dict[str, dict[str, Any]] = {}
    f1s: list[float] = []
    for c in class_list:
        hits = sum(1 for t, p in zip(y_true, y_pred, strict=True) if t == c and p == c)
        n_pred = sum(1 for p in y_pred if p == c)
        n_true = sum(1 for t in y_true if t == c)
        prf = _prf(hits, n_pred, n_true)
        per_class[c] = {**prf, "support": n_true, "predicted": n_pred}
        if n_true:
            f1s.append(prf["f1"])
    result: dict[str, Any] = {
        "n": n,
        "accuracy": round(correct / n, 4) if n else 0.0,
        "macro_f1": round(sum(f1s) / len(f1s), 4) if f1s else 0.0,
        "per_class": per_class,
        "classes": class_list,
        "decision_rule": "argmax over class scores",
        "auroc": None,
        "auroc_definition": None,
    }
    if scores is not None and n:
        if len(scores) != n or any(len(row) != len(class_list) for row in scores):
            raise ValueError("scores must be one row per record with one column per class")
        if len(class_list) == 2:
            pos = class_list[-1]
            result["auroc"] = auroc([1 if t == pos else 0 for t in y_true], [row[-1] for row in scores])
            result["auroc_definition"] = f"binary AUROC with positive class {pos!r} (last class in the list)"
        else:
            values = []
            for k, c in enumerate(class_list):
                a = auroc([1 if t == c else 0 for t in y_true], [row[k] for row in scores])
                if a is not None:
                    values.append(a)
            result["auroc"] = round(sum(values) / len(values), 4) if values else None
            result["auroc_definition"] = "macro-averaged one-vs-rest AUROC over classes present in the split"
    return result


def majority_baseline(
    train_records: Sequence[Mapping[str, Any]],
    eval_records: Sequence[Mapping[str, Any]],
    classes: Sequence[str],
) -> dict[str, Any]:
    """Predict the most frequent training class for every evaluation record (EVAL11)."""
    counts: dict[str, int] = {}
    for r in train_records:
        counts[r["label"]] = counts.get(r["label"], 0) + 1
    majority = max(sorted(counts), key=counts.__getitem__)
    metrics = classification_metrics(
        [r["label"] for r in eval_records], [majority] * len(eval_records), None, classes
    )
    return {"baseline": "majority-class", "predicted_label": majority, **metrics}


def library_size_baseline(
    train_records: Sequence[Mapping[str, Any]],
    eval_records: Sequence[Mapping[str, Any]],
    classes: Sequence[str],
) -> dict[str, Any]:
    """Threshold on a cell's total counts, fitted on the training split only (binary tasks).

    Library size is the first thing that separates cells in a badly designed single-cell experiment,
    so it is the baseline worth ruling out. The threshold and the class direction are chosen to
    maximise training accuracy; the evaluation split is never touched during fitting (SPL8). On the
    tutorial sample every cell carries the same total counts by construction, so this baseline is
    expected to sit at chance -- which is the point: it shows the model is reading the binned
    expression profile, not how much RNA was captured.
    """
    class_list = list(classes)
    if len(class_list) != 2:
        raise ValueError("library_size_baseline is defined for binary tasks only")
    lo, hi = class_list

    def total(record: Mapping[str, Any]) -> float:
        return float(sum(record["counts"].values()))

    train_x = [total(r) for r in train_records]
    train_y = [r["label"] for r in train_records]
    best = (-1.0, 0.0, True)  # accuracy, threshold, high_is_hi
    for t in sorted(set(train_x)):
        for high_is_hi in (True, False):
            pred = [(hi if (x >= t) == high_is_hi else lo) for x in train_x]
            acc = sum(p == y for p, y in zip(pred, train_y, strict=True)) / len(train_y)
            if acc > best[0]:
                best = (acc, t, high_is_hi)
    _, threshold, high_is_hi = best
    eval_x = [total(r) for r in eval_records]
    eval_pred = [(hi if (x >= threshold) == high_is_hi else lo) for x in eval_x]
    span = max(eval_x) - min(eval_x) or 1.0
    normalised = [(x - min(eval_x)) / span for x in eval_x]
    scores = [[1.0 - x, x] if high_is_hi else [x, 1.0 - x] for x in normalised]
    metrics = classification_metrics([r["label"] for r in eval_records], eval_pred, scores, class_list)
    return {
        "baseline": "library-size threshold",
        "threshold": round(threshold, 2),
        "rule": f"predict {hi!r} when total counts {'>=' if high_is_hi else '<'} {threshold:.0f}",
        "train_accuracy": round(best[0], 4),
        **metrics,
    }

**Module 2/3:** `src/scgpt_single_cell_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""scGPT (`tdc/scGPT`) DIMER pipeline: verified snapshot, single-cell expression embeddings,
masked-expression prediction, and bounded cell-state classification fine-tuning with a portable
adapter.

The checkpoint is the scGPT *whole-human* encoder (Cui et al., Nature Methods 2024; 33 million
cells) as repackaged in safetensors by Therapeutics Data Commons: 12 post-norm transformer layers of
width 512 with 8 heads, a 60,697-gene vocabulary, a gene-token encoder, a continuous value encoder,
and the masked-expression decoder. Three things about this row are stated rather than hidden:

* **The architecture is re-implemented here on plain torch modules.** The Hub repository ships no
  model code (`config.json` declares `model_type: "scgpt"` with no `auto_map`), and the packager's
  own loader lives in the PyTDC package with flash-attention and transformers as dependencies. This
  module rebuilds the encoder from `torch.nn.TransformerEncoder` and three small blocks whose
  parameter names match the checkpoint exactly, so the safetensors file loads with `strict=True`.
  It follows the **original** scGPT `model.py` (bowang-lab/scGPT @ `cebd6fae`): the value encoder
  clamps at 512 and applies a ReLU between its two linears, which the PyTDC port omits.
* **The gene vocabulary is not in the Hub repository.** The packager fetches it from Harvard
  Dataverse at runtime; this manifest lists it as a fourth entry with that persistent file id as its
  source and its SHA-256, so it is verified like every other file.
* **Inputs are binned, not raw.** scGPT reads expression as per-cell quantile bins (51 bins, zeros
  stay zero) after total-count normalisation and log1p; the packager's example feeds raw values,
  which is off-distribution. The binning here mirrors `scgpt/preprocess.py` deterministically.

Everything model-related is imported lazily so that snapshot verification and input validation run
(and can refuse) before `torch` is imported (fleet RTM-001). No transformers, no remote code.
"""

from __future__ import annotations

import hashlib
import json
import math
import warnings
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "tdc/scGPT"
MODEL_REVISION = "acf749f35bf5c0b00633838f02588272ed0d9911"
MODEL_LICENSE = "mit"
MODEL_KEY = "scgpt"
ARTIFACT_FORMAT = "org.valcorza.scgpt-single-cell.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
CONFIG_NAME = "config.json"
WEIGHTS_NAME = "model.safetensors"
VOCAB_NAME = "vocab.json"
# The vocabulary's persistent source (TDC `scgpt_vocab`, Harvard Dataverse datafile 10809431). It is
# the only manifest entry that is not a Hub file; `stage_missing_files` fetches it from here.
VOCAB_SOURCE_URL = "https://dataverse.harvard.edu/api/access/datafile/10809431"

# Architecture facts from the pinned config.json; asserted against the file at load time.
EMBSIZE = 512
NLAYERS = 12
NHEAD = 8
D_HID = 512
MAX_SEQ_LEN = 1536
VOCAB_SIZE = 60697
SPECIAL_TOKENS = ("<pad>", "<cls>", "<eoc>")
# scGPT input convention: per-cell quantile binning into N_BINS after normalisation, `<cls>` first
# with value 0, masked positions carry MASK_VALUE, and the value encoder clamps at VALUE_CLAMP.
N_BINS = 51
TARGET_SUM = 10_000.0
MASK_VALUE = -1.0
VALUE_CLAMP = 512
BINNING_SEED = 42  # seeds the tie-spreading in per-cell quantile binning (see bin_expression)
# Ceilings.
MAX_GENES_PER_CELL = MAX_SEQ_LEN - 1  # one position is the <cls> token
MAX_CELLS_PER_CALL = 32
MIN_DETECTED_GENES = 10  # fewer measured genes cannot be binned into a meaningful profile


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    listed = {entry["path"] for entry in manifest["files"]}
    for required in (CONFIG_NAME, WEIGHTS_NAME, VOCAB_NAME):
        if required not in listed:
            raise ValueError(f"manifest does not list {required}; refusing to proceed")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the snapshot against its DIMER manifest (size + SHA-256 of every listed file,
    including the Dataverse-sourced vocabulary)."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _verify_manifest(root, MODEL_ID, MODEL_REVISION)


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file: Hub files at the pinned revision, the vocabulary from its
    persistent Dataverse file id. The caller's `verify_snapshot` checks the digest afterwards."""
    if relative_path == VOCAB_NAME:
        import urllib.request

        request = urllib.request.Request(
            VOCAB_SOURCE_URL, headers={"User-Agent": "scgpt-single-cell-pipeline"}
        )
        with urllib.request.urlopen(request, timeout=120) as response:
            data = response.read()
        (root / VOCAB_NAME).write_bytes(data)
        return
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest, the config
    and the vocabulary but git-ignores the safetensors file). `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


INPUT_SCHEMA: dict[str, Any] = {
    "input": "1..MAX_CELLS_PER_CALL cells, each a {gene symbol: count} mapping of raw or normalised counts",
    "cells": [1, MAX_CELLS_PER_CALL],
    "genes_per_cell": [MIN_DETECTED_GENES, MAX_GENES_PER_CELL],
    "validation": (
        "type, finiteness and non-negativity of counts, a minimum number of detected genes, and gene-symbol "
        "resolution against the pinned 60,697-symbol vocabulary. Nothing checks that a profile is a real "
        "cell, that counts are on a consistent scale across cells, or that symbols follow one nomenclature"
    ),
    "preprocessing": (
        "genes with zero counts or unknown symbols are dropped and reported; the rest are normalised to "
        f"{TARGET_SUM:g} total counts, log1p-transformed, quantile-binned per cell into {N_BINS} bins (zeros "
        "stay 0), truncated to the most expressed MAX_GENES_PER_CELL genes, and fed as (gene token, bin) "
        "pairs after a <cls> token with value 0; the cell embedding is the <cls> output (512 floats)"
    ),
}


@dataclass(frozen=True)
class GeneVocabulary:
    """Gene symbol -> token id, from the pinned vocab.json; special tokens resolved by name."""

    tokens: dict[str, int]
    pad_id: int
    cls_id: int
    eoc_id: int

    @property
    def gene_symbols(self) -> list[str]:
        return [s for s in self.tokens if s not in SPECIAL_TOKENS]

    def resolve(self, symbol: str) -> int | None:
        """Exact match first, then a case-insensitive upper-case match; None when unknown."""
        if symbol in self.tokens:
            return self.tokens[symbol]
        upper = symbol.upper()
        return self.tokens.get(upper) if upper not in SPECIAL_TOKENS else None


def load_gene_vocabulary(path: str | Path | None = None) -> GeneVocabulary:
    """Read the pinned vocab.json (60,697 entries: 60,694 gene symbols + <pad>, <cls>, <eoc>)."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    raw = json.loads((root / VOCAB_NAME).read_text(encoding="utf-8"))
    if not isinstance(raw, dict) or len(raw) != VOCAB_SIZE:
        raise ValueError(
            f"vocab.json must map {VOCAB_SIZE} symbols to ids, got {type(raw).__name__} of {len(raw)}"
        )
    missing = [t for t in SPECIAL_TOKENS if t not in raw]
    if missing:
        raise ValueError(f"vocab.json lacks special tokens {missing}")
    ids = sorted(int(v) for v in raw.values())
    if ids != list(range(VOCAB_SIZE)):
        raise ValueError("vocab.json ids must be exactly 0..VOCAB_SIZE-1 with no gaps or duplicates")
    tokens = {str(k): int(v) for k, v in raw.items()}
    return GeneVocabulary(tokens, tokens["<pad>"], tokens["<cls>"], tokens["<eoc>"])


def _check_cells(cells: Any, names: Any = None) -> tuple[list[dict[str, float]], list[str]]:
    """Raise TypeError/ValueError naming the first violated rule; return (cells, ids).

    ``embed``, ``predict_masked``, ``classify`` and ``validate_inputs`` all route through this
    function so their acceptance criteria cannot diverge.
    """
    if isinstance(cells, Mapping) or not isinstance(cells, Sequence) or isinstance(cells, str | bytes):
        raise TypeError("cells must be a list of {gene: count} mappings (one mapping per cell)")
    if not 1 <= len(cells) <= MAX_CELLS_PER_CALL:
        raise ValueError(f"cells must hold 1..{MAX_CELLS_PER_CALL} items, got {len(cells)}")
    checked: list[dict[str, float]] = []
    for i, cell in enumerate(cells):
        if not isinstance(cell, Mapping):
            raise TypeError(f"cells[{i}] must be a mapping of gene to count, got {type(cell).__name__}")
        if not cell:
            raise ValueError(f"cells[{i}] is empty")
        counts: dict[str, float] = {}
        for gene, value in cell.items():
            if not isinstance(gene, str) or not gene.strip():
                raise TypeError(f"cells[{i}] has a non-string gene key {gene!r}")
            if isinstance(value, bool) or not isinstance(value, int | float):
                raise TypeError(f"cells[{i}][{gene!r}] must be a number, got {type(value).__name__}")
            if value < 0 or value != value or value in (float("inf"), float("-inf")):
                raise ValueError(f"cells[{i}][{gene!r}] must be a finite non-negative count, got {value!r}")
            counts[gene] = float(value)
        detected = sum(1 for v in counts.values() if v > 0)
        if detected < MIN_DETECTED_GENES:
            raise ValueError(
                f"cells[{i}] has {detected} detected gene(s); at least {MIN_DETECTED_GENES} are required "
                "to bin an expression profile"
            )
        checked.append(counts)
    if names is None:
        ids = [f"cell-{i}" for i in range(len(checked))]
    else:
        if isinstance(names, str | bytes) or not isinstance(names, Sequence) or len(names) != len(checked):
            raise ValueError("names must be a list with exactly one id per cell")
        ids = [str(n) for n in names]
        if len(set(ids)) != len(ids):
            raise ValueError("names must be unique")
    return checked, ids


def _quantile(sorted_values: Sequence[float], q: float) -> float:
    """numpy's default (linear) quantile on an ascending list, without importing numpy."""
    if not sorted_values:
        raise ValueError("quantile of an empty sequence")
    pos = q * (len(sorted_values) - 1)
    lo = int(math.floor(pos))
    hi = min(lo + 1, len(sorted_values) - 1)
    frac = pos - lo
    return sorted_values[lo] * (1.0 - frac) + sorted_values[hi] * frac


def bin_expression(
    counts: Mapping[str, float],
    vocabulary: GeneVocabulary,
    *,
    n_bins: int = N_BINS,
    target_sum: float = TARGET_SUM,
    max_genes: int = MAX_GENES_PER_CELL,
    seed: int = BINNING_SEED,
) -> dict[str, Any]:
    """scGPT input encoding of one cell: tokens, per-cell quantile bins, and what was dropped.

    Mirrors `scgpt/preprocess.py`: detected genes are normalised to `target_sum` total counts and
    log1p-transformed, then the non-zero values are digitised against `n_bins - 1` quantile edges
    (bins 1..n_bins-1; zeros are never tokenised). Upstream's `_digitize(side="both")` spreads tied
    values **uniformly at random** between their left and right bin — sparse count data has many
    ties (every count-of-1 gene shares one value), and the checkpoint was trained on that spread —
    so this function does the same with a seeded generator: identical input and seed give identical
    bins, and the distribution matches upstream's. Genes absent from the vocabulary are dropped and
    reported rather than mapped to a real gene (the packager's tokenizer maps unknowns to id 0,
    which is the gene A1BG). Cells with more than `max_genes` detected genes keep the most
    expressed ones and report the truncation.
    """
    import random

    rng = random.Random(seed)
    unknown: list[str] = []
    resolved: dict[int, float] = {}
    symbols: dict[int, str] = {}
    for gene, value in counts.items():
        if value <= 0:
            continue
        token = vocabulary.resolve(gene)
        if token is None:
            unknown.append(gene)
            continue
        resolved[token] = resolved.get(token, 0.0) + value
        symbols.setdefault(token, gene)
    if not resolved:
        raise ValueError(
            "no gene in this cell could be encoded: none of the detected genes is in the scGPT vocabulary "
            "(check that gene identifiers are human gene symbols such as GAPDH, not Ensembl ids)"
        )
    library = sum(resolved.values())
    logged = {tok: math.log1p(v / library * target_sum) for tok, v in resolved.items()}
    ascending = sorted(logged.values())
    edges = [_quantile(ascending, k / (n_bins - 2)) for k in range(n_bins - 1)]

    def digitize(x: float) -> int:
        # numpy.digitize(x, edges): edges <= x (left) and edges < x (right); ties spread between them.
        left = sum(1 for e in edges if e <= x)
        right = sum(1 for e in edges if e < x)
        return int(math.ceil(rng.random() * (left - right) + right)) if left != right else left

    # Digitise in a fixed order (ascending token id) so the seeded spread is reproducible.
    binned = {tok: max(1, min(n_bins - 1, digitize(logged[tok]))) for tok in sorted(logged)}
    ranked = sorted(binned.items(), key=lambda item: (-item[1], -logged[item[0]], item[0]))
    kept = ranked[:max_genes]
    return {
        "tokens": [vocabulary.cls_id] + [tok for tok, _ in kept],
        "values": [0.0] + [float(b) for _, b in kept],
        "gene_symbols": [symbols[tok] for tok, _ in kept],
        "n_detected": sum(1 for v in counts.values() if v > 0),
        "n_encoded": len(resolved),
        "n_kept": len(kept),
        "n_truncated": max(0, len(ranked) - len(kept)),
        "unknown_genes": sorted(set(unknown)),
        "library_size": library,
        "n_bins": n_bins,
        "bin_edges": edges,
    }


def validate_inputs(
    cells: Sequence[Mapping[str, float]],
    vocabulary: GeneVocabulary,
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: encode every cell and return the input manifest (schema, observations, verdict).

    Rejection is reported by raising exactly as ``embed``/``classify`` would.
    """
    checked, ids = _check_cells(cells, names)
    rows = []
    for cid, counts in zip(ids, checked, strict=True):
        encoded = bin_expression(counts, vocabulary)
        rows.append(
            {
                "id": cid,
                "detected_genes": encoded["n_detected"],
                "encoded_genes": encoded["n_encoded"],
                "tokens_kept": encoded["n_kept"],
                "genes_truncated": encoded["n_truncated"],
                "unknown_genes": len(encoded["unknown_genes"]),
                "library_size": encoded["library_size"],
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": rows,
        "n_cells": len(checked),
        "max_tokens_observed": 1 + max(row["tokens_kept"] for row in rows),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "requires_remote_code": False,
    }


def _softmax(logits: Sequence[float]) -> list[float]:
    top = max(logits)
    exps = [math.exp(v - top) for v in logits]
    total = sum(exps)
    return [v / total for v in exps]


def _pearson(a: Sequence[float], b: Sequence[float]) -> float | None:
    n = len(a)
    if n < 2:
        return None
    ma, mb = sum(a) / n, sum(b) / n
    cov = sum((x - ma) * (y - mb) for x, y in zip(a, b, strict=True))
    va = sum((x - ma) ** 2 for x in a)
    vb = sum((y - mb) ** 2 for y in b)
    if va == 0 or vb == 0:
        return None
    return cov / math.sqrt(va * vb)


def build_model(config: Mapping[str, Any], vocabulary: GeneVocabulary) -> Any:
    """The scGPT encoder on plain torch modules, named so the pinned checkpoint loads strictly.

    Follows bowang-lab/scGPT `model.py` (`TransformerModel` with `input_emb_style="continuous"`,
    `cell_emb_style="cls"`, post-norm `nn.TransformerEncoderLayer` with ReLU, `ContinuousValueEncoder`
    with clamp and ReLU, `ExprDecoder` with LeakyReLU). torch is imported here, not at module import.
    """
    import torch
    from torch import nn

    class ScGPTEncoder(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            embsize, d_hid = int(config["embsize"]), int(config["d_hid"])
            dropout = float(config.get("dropout", 0.0))
            self.gene_encoder = nn.ModuleDict(
                {
                    "embedding": nn.Embedding(
                        int(config["vocab_size"]), embsize, padding_idx=vocabulary.pad_id
                    ),
                    "enc_norm": nn.LayerNorm(embsize),
                }
            )
            self.value_encoder = nn.ModuleDict(
                {
                    "linear1": nn.Linear(1, embsize),
                    "linear2": nn.Linear(embsize, embsize),
                    "norm": nn.LayerNorm(embsize),
                    "dropout": nn.Dropout(dropout),
                }
            )
            layer = nn.TransformerEncoderLayer(
                d_model=embsize,
                nhead=int(config["nhead"]),
                dim_feedforward=d_hid,
                dropout=dropout,
                activation="relu",
                batch_first=True,
                norm_first=False,
            )
            self.transformer = nn.TransformerEncoder(
                layer, num_layers=int(config["nlayers"]), enable_nested_tensor=False
            )
            self.expr_decoder = nn.ModuleDict(
                {
                    "fc": nn.Sequential(
                        nn.Linear(embsize, embsize),
                        nn.LeakyReLU(),
                        nn.Linear(embsize, embsize),
                        nn.LeakyReLU(),
                        nn.Linear(embsize, 1),
                    )
                }
            )
            self.value_clamp = VALUE_CLAMP

        def forward(self, input_ids: Any, values: Any, padding_mask: Any) -> dict[str, Any]:
            gene = self.gene_encoder["enc_norm"](self.gene_encoder["embedding"](input_ids))
            v = torch.clamp(values.unsqueeze(-1), max=self.value_clamp)
            v = torch.relu(self.value_encoder["linear1"](v))
            v = self.value_encoder["norm"](self.value_encoder["linear2"](v))
            v = self.value_encoder["dropout"](v)
            hidden = self.transformer(gene + v, src_key_padding_mask=padding_mask)
            return {"cell_emb": hidden[:, 0], "pred": self.expr_decoder["fc"](hidden).squeeze(-1)}

    return ScGPTEncoder()


@dataclass
class ScGPTPipeline:
    """scGPT pipeline: `embed` and `predict_masked` always; `classify` after `adapt` or `from_artifact`."""

    _embedder: Callable[[list[dict[str, float]]], list[list[float]]]
    vocabulary: GeneVocabulary
    device: str
    load_warnings: list[str] = field(default_factory=list)
    classes: list[str] = field(default_factory=list)
    _classifier: Callable[[list[dict[str, float]]], list[list[float]]] | None = None
    model: Any = None
    config: dict[str, Any] = field(default_factory=dict)
    classifier_model: Any = None
    weights_dir: Path | None = None
    adaptation: dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ScGPTPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(f"no snapshot manifest at {root} and allow_download={allow_download}")
        # Stage and verify before importing model libraries (RTM-001).
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        config = json.loads((root / CONFIG_NAME).read_text(encoding="utf-8"))
        expected = {
            "embsize": (config.get("embsize"), EMBSIZE),
            "nlayers": (config.get("nlayers"), NLAYERS),
            "nhead": (config.get("nhead"), NHEAD),
            "d_hid": (config.get("d_hid"), D_HID),
            "max_seq_len": (config.get("max_seq_len"), MAX_SEQ_LEN),
            "vocab_size": (config.get("vocab_size"), VOCAB_SIZE),
            "input_emb_style": (config.get("input_emb_style"), "continuous"),
            "cell_emb_style": (config.get("cell_emb_style"), "cls"),
            "norm_scheme": (config.get("norm_scheme"), "post"),
            "explicit_zero_prob": (config.get("explicit_zero_prob"), False),
        }
        drift = {k: v for k, v in expected.items() if v[0] != v[1]}
        if drift:
            raise ValueError(f"pinned config.json disagrees with the package constants: {drift}")
        vocabulary = load_gene_vocabulary(root)

        import torch
        from safetensors.torch import load_file

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model = build_model(config, vocabulary)
            state = load_file(str(root / WEIGHTS_NAME))
            model.load_state_dict(state, strict=True)
            del state
        model = model.to(resolved_device).eval()
        messages = [f"{w.category.__name__}: {w.message}" for w in caught]
        pipe = cls(
            cls._make_embedder(model, vocabulary, resolved_device), vocabulary, resolved_device, messages
        )
        pipe.model, pipe.config, pipe.weights_dir = model, dict(config), root
        return pipe

    # -- backends ---------------------------------------------------------------------------------

    @staticmethod
    def _batch(
        cells: list[dict[str, float]],
        vocabulary: GeneVocabulary,
        device: str,
        mask: Mapping[int, set[int]] | None = None,
    ) -> tuple[Any, Any, Any, list[dict[str, Any]]]:
        """Encode cells, pad to the longest, and return (input_ids, values, padding_mask, encodings).

        `mask` maps a cell index to the set of *positions* whose value is replaced by MASK_VALUE.
        """
        import torch

        encoded = [bin_expression(c, vocabulary) for c in cells]
        width = max(len(e["tokens"]) for e in encoded)
        ids = torch.full((len(encoded), width), vocabulary.pad_id, dtype=torch.long)
        vals = torch.zeros((len(encoded), width), dtype=torch.float32)
        pad = torch.ones((len(encoded), width), dtype=torch.bool)
        for i, e in enumerate(encoded):
            n = len(e["tokens"])
            ids[i, :n] = torch.tensor(e["tokens"], dtype=torch.long)
            row = list(e["values"])
            for pos in (mask or {}).get(i, set()):
                row[pos] = MASK_VALUE
            vals[i, :n] = torch.tensor(row, dtype=torch.float32)
            pad[i, :n] = False
        return ids.to(device), vals.to(device), pad.to(device), encoded

    @classmethod
    def _make_embedder(
        cls, model: Any, vocabulary: GeneVocabulary, device: str
    ) -> Callable[[list[dict[str, float]]], list[list[float]]]:
        import torch

        def embedder(cells: list[dict[str, float]]) -> list[list[float]]:
            ids, vals, pad, _ = cls._batch(cells, vocabulary, device)
            # no_grad, not inference_mode: tensors produced here must stay usable by a later
            # training epoch that shares this module.
            with torch.no_grad():
                out = model(ids, vals, pad)
            return out["cell_emb"].float().cpu().tolist()

        return embedder

    @classmethod
    def _make_classifier(
        cls, clf: Any, vocabulary: GeneVocabulary, device: str
    ) -> Callable[[list[dict[str, float]]], list[list[float]]]:
        import torch

        def classifier(cells: list[dict[str, float]]) -> list[list[float]]:
            ids, vals, pad, _ = cls._batch(cells, vocabulary, device)
            with torch.no_grad():
                logits = clf(ids, vals, pad)
            return logits.float().cpu().tolist()

        return classifier

    # -- public stages ----------------------------------------------------------------------------

    def embed(
        self, cells: Sequence[Mapping[str, float]], *, names: Sequence[str] | None = None
    ) -> dict[str, Any]:
        """`<cls>` output of the encoder per cell (EMBSIZE floats each); order-invariant over genes."""
        checked, ids = _check_cells(cells, names)
        vectors = self._embedder(checked)
        if len(vectors) != len(checked) or any(len(v) != EMBSIZE for v in vectors):
            raise RuntimeError("backend returned embeddings of the wrong shape")
        encodings = [bin_expression(c, self.vocabulary) for c in checked]
        return {
            "ids": ids,
            "embeddings": [[float(x) for x in v] for v in vectors],
            "dimension": EMBSIZE,
            "pooling": "<cls> token output of the last encoder layer (cell_emb_style 'cls')",
            "unit": "one vector per cell; representations, not cell-state predictions",
            "tokens": [e["n_kept"] + 1 for e in encodings],
            "unknown_genes": [len(e["unknown_genes"]) for e in encodings],
            "n_cells": len(checked),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def predict_masked(
        self,
        cells: Sequence[Mapping[str, float]],
        *,
        names: Sequence[str] | None = None,
        mask_fraction: float = 0.15,
        seed: int = 42,
    ) -> dict[str, Any]:
        """Masked-expression prediction — scGPT's pretraining objective, run as a functional check.

        A seeded fraction of each cell's gene positions receives MASK_VALUE; the decoder predicts a
        bin for every position, and the masked ones are compared with the true bins (mean absolute
        error and Pearson correlation per cell). A correct re-implementation of the encoder should
        predict masked bins far better than chance; a wrong one produces noise.
        """
        if self.model is None:
            raise RuntimeError("predict_masked requires a pipeline built by from_pretrained")
        if not 0.0 < float(mask_fraction) <= 0.5:
            raise ValueError("mask_fraction must be in (0, 0.5]")
        import random

        import torch

        checked, ids = _check_cells(cells, names)
        rng = random.Random(seed)
        pre = [bin_expression(c, self.vocabulary) for c in checked]
        mask: dict[int, set[int]] = {}
        for i, e in enumerate(pre):
            positions = list(range(1, len(e["tokens"])))  # never mask <cls>
            k = max(1, int(round(len(positions) * float(mask_fraction))))
            mask[i] = set(rng.sample(positions, k))
        input_ids, vals, pad, encoded = self._batch(checked, self.vocabulary, self.device, mask)
        with torch.no_grad():
            out = self.model(input_ids, vals, pad)
        pred = out["pred"].float().cpu().tolist()
        rows = []
        for i, (cid, e) in enumerate(zip(ids, encoded, strict=True)):
            positions = sorted(mask[i])
            truth = [e["values"][p] for p in positions]
            guess = [pred[i][p] for p in positions]
            mae = sum(abs(t - g) for t, g in zip(truth, guess, strict=True)) / len(positions)
            rows.append(
                {
                    "id": cid,
                    "masked_positions": len(positions),
                    "mean_absolute_error_bins": round(mae, 4),
                    "pearson": (None if (r := _pearson(truth, guess)) is None else round(r, 4)),
                    "mean_true_bin": round(sum(truth) / len(truth), 3),
                    "mean_predicted_bin": round(sum(guess) / len(guess), 3),
                }
            )
        return {
            "cells": rows,
            "mask_fraction": float(mask_fraction),
            "mask_value": MASK_VALUE,
            "n_bins": N_BINS,
            "note": (
                "functional check of the re-implemented encoder against scGPT's masked-expression objective; "
                "bins are per-cell quantiles, so chance-level prediction sits near the mean bin"
            ),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def predict_masked_genes(
        self,
        cells: Sequence[Mapping[str, float]],
        genes: Sequence[str],
        *,
        names: Sequence[str] | None = None,
    ) -> dict[str, Any]:
        """Mask the named genes in each cell and report the decoder's mean predicted bin against the
        true mean bin — the lineage-conditioning probe: does the decoder predict a programme's genes
        high where the rest of the cell expresses that programme?"""
        if self.model is None:
            raise RuntimeError("predict_masked_genes requires a pipeline built by from_pretrained")
        if isinstance(genes, str) or not genes:
            raise ValueError("genes must be a non-empty list of gene symbols")
        import torch

        checked, ids = _check_cells(cells, names)
        wanted = {str(g) for g in genes}
        pre = [bin_expression(c, self.vocabulary) for c in checked]
        mask: dict[int, set[int]] = {}
        for i, e in enumerate(pre):
            mask[i] = {k + 1 for k, symbol in enumerate(e["gene_symbols"]) if symbol in wanted}
            if not mask[i]:
                raise ValueError(f"{ids[i]}: none of the requested genes is detected in this cell")
        input_ids, vals, pad, encoded = self._batch(checked, self.vocabulary, self.device, mask)
        with torch.no_grad():
            pred = self.model(input_ids, vals, pad)["pred"].float().cpu().tolist()
        rows = []
        for i, (cid, e) in enumerate(zip(ids, encoded, strict=True)):
            positions = sorted(mask[i])
            rows.append(
                {
                    "id": cid,
                    "masked_genes": len(positions),
                    "predicted_mean_bin": round(sum(pred[i][p] for p in positions) / len(positions), 3),
                    "true_mean_bin": round(sum(e["values"][p] for p in positions) / len(positions), 3),
                }
            )
        n = len(rows)
        return {
            "cells": rows,
            "predicted_mean_bin": round(sum(r["predicted_mean_bin"] for r in rows) / n, 3),
            "true_mean_bin": round(sum(r["true_mean_bin"] for r in rows) / n, 3),
            "genes_requested": len(wanted),
            "mask_value": MASK_VALUE,
        }

    def nearest_centroid_evaluate(
        self,
        train_records: Sequence[Mapping[str, Any]],
        eval_records: Sequence[Mapping[str, Any]],
        *,
        classes: Sequence[str] | None = None,
    ) -> dict[str, Any]:
        """The frozen embedding's own separability, with no gradient training: centre the `<cls>`
        embeddings on the training mean, fit one centroid per class on the training split only
        (SPL8), and assign each evaluation cell to the nearest centroid by cosine.

        scGPT's raw `<cls>` vectors share a large common component (cosine ~0.95 between unrelated
        cells), so centring is what makes the geometry readable; the training mean is the only
        fitted state besides the centroids. This is the zero-training baseline the adapted head must
        be read against.
        """
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        manifest = validate_dataset(train_records, classes=classes)
        class_list = list(manifest["classes"])
        validate_dataset(eval_records, classes=class_list, min_records=2, min_per_class=1)

        def embed_all(records: Sequence[Mapping[str, Any]]) -> list[list[float]]:
            out: list[list[float]] = []
            for start in range(0, len(records), MAX_CELLS_PER_CALL):
                chunk = records[start : start + MAX_CELLS_PER_CALL]
                out.extend(
                    self.embed([r["counts"] for r in chunk], names=[r["id"] for r in chunk])["embeddings"]
                )
            return out

        train_emb = embed_all(train_records)
        mean = [sum(e[d] for e in train_emb) / len(train_emb) for d in range(EMBSIZE)]

        def centred(e: Sequence[float]) -> list[float]:
            return [x - m for x, m in zip(e, mean, strict=True)]

        def cosine(a: Sequence[float], b: Sequence[float]) -> float:
            na = math.sqrt(sum(x * x for x in a)) or 1.0
            nb = math.sqrt(sum(x * x for x in b)) or 1.0
            return sum(x * y for x, y in zip(a, b, strict=True)) / (na * nb)

        centroids: dict[str, list[float]] = {}
        for c in class_list:
            rows = [centred(e) for e, r in zip(train_emb, train_records, strict=True) if r["label"] == c]
            centroids[c] = [sum(r[d] for r in rows) / len(rows) for d in range(EMBSIZE)]
        predicted: list[str] = []
        scores: list[list[float]] = []
        for e in embed_all(eval_records):
            sims = [cosine(centred(e), centroids[c]) for c in class_list]
            predicted.append(class_list[max(range(len(sims)), key=sims.__getitem__)])
            scores.append(_softmax([s * 10.0 for s in sims]))
        metrics = classification_metrics([r["label"] for r in eval_records], predicted, scores, class_list)
        return {
            "baseline": "nearest class centroid on centred frozen <cls> embeddings (no training)",
            "fitted_on": "training split only (mean and centroids)",
            **metrics,
        }

    def classify(
        self, cells: Sequence[Mapping[str, float]], *, names: Sequence[str] | None = None
    ) -> dict[str, Any]:
        """Class scores and argmax label per cell; requires a prior `adapt` or `from_artifact`."""
        if self._classifier is None or not self.classes:
            raise RuntimeError(
                "classify requires an adapted head: call adapt(...) or load from_artifact(...) first"
            )
        checked, ids = _check_cells(cells, names)
        logits = self._classifier(checked)
        predictions = []
        for cid, row in zip(ids, logits, strict=True):
            if len(row) != len(self.classes):
                raise RuntimeError("backend returned a logits row that does not match the class list")
            scores = _softmax(row)
            best = max(range(len(scores)), key=scores.__getitem__)
            predictions.append(
                {
                    "id": cid,
                    "label": self.classes[best],
                    "score": scores[best],
                    "scores": dict(zip(self.classes, scores, strict=True)),
                }
            )
        return {
            "predictions": predictions,
            "classes": list(self.classes),
            "decision_rule": (
                "argmax over softmax(logits); scores are softmax outputs, not calibrated probabilities"
            ),
            "n_cells": len(checked),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "adaptation": dict(self.adaptation),
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Held-out cell-state classification metrics (see metrics.classification_metrics)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        validate_dataset(records, classes=self.classes, min_records=2, min_per_class=1)
        predicted: list[str] = []
        scores: list[list[float]] = []
        for start in range(0, len(records), MAX_CELLS_PER_CALL):
            chunk = records[start : start + MAX_CELLS_PER_CALL]
            result = self.classify([r["counts"] for r in chunk], names=[r["id"] for r in chunk])
            for p in result["predictions"]:
                predicted.append(p["label"])
                scores.append([p["scores"][c] for c in self.classes])
        return classification_metrics([r["label"] for r in records], predicted, scores, self.classes)

    def _build_classifier(self, n_classes: int) -> Any:
        """A fresh copy of the encoder with a linear head on its `<cls>` output."""
        import copy

        import torch

        class CellClassifier(torch.nn.Module):
            def __init__(self, encoder: Any, n: int) -> None:
                super().__init__()
                self.encoder = encoder
                self.head = torch.nn.Linear(EMBSIZE, n)

            def forward(self, input_ids: Any, values: Any, padding_mask: Any) -> Any:
                return self.head(self.encoder(input_ids, values, padding_mask)["cell_emb"])

        return CellClassifier(copy.deepcopy(self.model), n_classes)

    def adapt(
        self,
        train_records: Sequence[Mapping[str, Any]],
        val_records: Sequence[Mapping[str, Any]] | None = None,
        *,
        classes: Sequence[str] | None = None,
        epochs: int = 4,
        learning_rate: float = 1e-4,
        batch_size: int = 8,
        trainable_layers: int = 1,
        weight_decay: float = 0.01,
        seed: int = 42,
    ) -> dict[str, Any]:
        """Bounded gradient fine-tuning of a cell-state classification head on the verified base.

        Copies the encoder, puts a newly initialised linear head over its `<cls>` output, freezes
        everything except the head and the last `trainable_layers` transformer layers, and runs
        AdamW for `epochs` passes. With `trainable_layers=0` the frozen encoder's output is computed
        once and only the head trains. Validation records are monitored per epoch only; the final
        epoch's weights are kept (no selection).
        """
        if self.model is None or self.weights_dir is None:
            raise RuntimeError("adapt requires a pipeline built by from_pretrained (no loaded base model)")
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not 1 <= int(epochs) <= 50:
            raise ValueError("epochs must be in 1..50 (tutorial-scale adaptation)")
        if not 1 <= int(batch_size) <= MAX_CELLS_PER_CALL:
            raise ValueError(f"batch_size must be in 1..{MAX_CELLS_PER_CALL}")
        if not 0 <= int(trainable_layers) <= NLAYERS:
            raise ValueError(f"trainable_layers must be in 0..{NLAYERS} (the encoder has that many)")
        train_manifest = validate_dataset(train_records, classes=classes)
        class_list = list(train_manifest["classes"])
        if val_records is not None:
            validate_dataset(val_records, classes=class_list, min_records=2, min_per_class=1)

        import random

        import torch

        random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        clf = self._build_classifier(len(class_list)).to(self.device)
        for p in clf.parameters():
            p.requires_grad = False
        layers = clf.encoder.transformer.layers
        for layer in layers[len(layers) - int(trainable_layers) :] if trainable_layers else []:
            for p in layer.parameters():
                p.requires_grad = True
        for p in clf.head.parameters():
            p.requires_grad = True
        trainable = [n for n, p in clf.named_parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in clf.parameters() if p.requires_grad)
        n_total = sum(p.numel() for p in clf.parameters())
        optimizer = torch.optim.AdamW(
            [p for p in clf.parameters() if p.requires_grad], lr=learning_rate, weight_decay=weight_decay
        )
        label_index = {c: i for i, c in enumerate(class_list)}
        examples = [(dict(r["counts"]), label_index[r["label"]]) for r in train_records]
        self.classes = class_list
        self.classifier_model = clf
        self._classifier = self._make_classifier(clf, self.vocabulary, self.device)
        loss_fn = torch.nn.CrossEntropyLoss()
        cached: list[Any] | None = None
        if not trainable_layers:
            clf.eval()
            cached = []
            with torch.no_grad():
                for start in range(0, len(examples), MAX_CELLS_PER_CALL):
                    ids, vals, pad, _ = self._batch(
                        [c for c, _ in examples[start : start + MAX_CELLS_PER_CALL]],
                        self.vocabulary,
                        self.device,
                    )
                    cached.extend(clf.encoder(ids, vals, pad)["cell_emb"])

        history: list[dict[str, Any]] = []
        for epoch in range(1, int(epochs) + 1):
            clf.train()
            order = list(range(len(examples)))
            random.shuffle(order)
            total_loss, n_batches = 0.0, 0
            for start in range(0, len(order), int(batch_size)):
                idx = order[start : start + int(batch_size)]
                labels = torch.tensor([examples[i][1] for i in idx], device=self.device)
                optimizer.zero_grad()
                if cached is not None:
                    logits = clf.head(torch.stack([cached[i] for i in idx]))
                else:
                    ids, vals, pad, _ = self._batch(
                        [examples[i][0] for i in idx], self.vocabulary, self.device
                    )
                    logits = clf(ids, vals, pad)
                loss = loss_fn(logits, labels)
                loss.backward()
                optimizer.step()
                total_loss += float(loss.item())
                n_batches += 1
            clf.eval()
            entry: dict[str, Any] = {
                "epoch": epoch,
                "train_loss": round(total_loss / max(1, n_batches), 6),
                "n_batches": n_batches,
            }
            if val_records:
                val = self.evaluate(val_records)
                entry["val_accuracy"] = val["accuracy"]
                entry["val_macro_f1"] = val["macro_f1"]
            history.append(entry)
        clf.eval()
        self.adaptation = {
            "method": "gradient fine-tuning (AdamW) of a linear head over the <cls> cell embedding"
            + (
                f" and the last {int(trainable_layers)} encoder layer(s)"
                if trainable_layers
                else " (frozen encoder; embeddings computed once)"
            ),
            "classes": class_list,
            "epochs": int(epochs),
            "learning_rate": float(learning_rate),
            "batch_size": int(batch_size),
            "weight_decay": float(weight_decay),
            "trainable_layers": int(trainable_layers),
            "seed": int(seed),
            "precision": "float32",
            "trainable_parameters": int(n_trainable),
            "total_parameters": int(n_total),
            "trainable_parameter_names": trainable,
            "train_records": len(train_records),
            "val_records": len(val_records) if val_records else 0,
            "selection": "final epoch kept; validation metrics are monitoring only",
            "history": history,
        }
        return dict(self.adaptation)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Export the trainable tensors as safetensors plus a JSON manifest binding them to the base."""
        if self.classifier_model is None or not self.classes:
            raise RuntimeError("save_artifact requires an adapted head (call adapt first)")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adaptation.get("trainable_parameter_names", []))
        state = self.classifier_model.state_dict()
        tensors = {k: v.detach().cpu().contiguous() for k, v in state.items() if k in names}
        if not tensors:
            raise RuntimeError("no trainable tensors recorded; nothing to export")
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path))
        digest = hashlib.sha256(weights_path.read_bytes()).hexdigest()
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"model_id": MODEL_ID, "model_revision": MODEL_REVISION, "license": MODEL_LICENSE},
            "requires_remote_code": False,
            "classes": list(self.classes),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": digest}
            ],
            "tensors": sorted(tensors),
            "serving_state_tensors": [],
            "adaptation": {k: v for k, v in self.adaptation.items() if k != "trainable_parameter_names"},
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Rebuild the classification head from an exported artifact (manifest verified before loading)."""
        if self.model is None or self.weights_dir is None:
            raise RuntimeError("load_artifact requires a pipeline built by from_pretrained")
        art = Path(artifact_dir)
        manifest_path = art / ARTIFACT_MANIFEST_NAME
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest not found: {manifest_path}")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("model_id"), base.get("model_revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError(f"artifact was trained on {base}, this package pins {MODEL_ID}@{MODEL_REVISION}")
        classes = [str(c) for c in manifest.get("classes", [])]
        if len(classes) < 2 or len(set(classes)) != len(classes):
            raise ValueError("artifact manifest must list at least two unique classes")
        for entry in manifest["files"]:
            fp = art / entry["path"]
            if not fp.is_file():
                raise FileNotFoundError(f"artifact file missing: {fp}")
            if fp.stat().st_size != entry["bytes"]:
                raise ValueError(f"{entry['path']}: size {fp.stat().st_size} != manifest {entry['bytes']}")
            if hashlib.sha256(fp.read_bytes()).hexdigest() != entry["sha256"]:
                raise ValueError(f"{entry['path']}: sha256 mismatch against the artifact manifest")
        from safetensors.torch import load_file

        clf = self._build_classifier(len(classes))
        tensors = load_file(str(art / ARTIFACT_WEIGHTS_NAME))
        if set(tensors) != set(manifest.get("tensors", [])):
            raise ValueError("artifact tensors do not match the names listed in its manifest")
        _missing, unexpected = clf.load_state_dict(tensors, strict=False)
        if unexpected:
            raise ValueError(
                f"artifact carries tensors the base architecture does not have: {sorted(unexpected)[:5]}"
            )
        clf = clf.to(self.device).eval()
        self.classes = classes
        self.classifier_model = clf
        self._classifier = self._make_classifier(clf, self.vocabulary, self.device)
        self.adaptation = {**manifest.get("adaptation", {}), "loaded_from_artifact": str(art)}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ScGPTPipeline:
        """Verified base snapshot + exported adapter, ready for `classify`."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 3/3:** `src/scgpt_single_cell_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic in-code sample cells and the labelled-dataset contract for cell-state classification.

The tutorial dataset is synthetic but built from **real human gene symbols and real lineage
programmes**: two sets of 24 canonical PBMC marker genes — a T-lymphocyte programme (CD3D, CD3E,
IL7R, TRAC, …) and a B-lymphocyte programme (CD79A, MS4A1, CD19, IGHM, …) — plus 300 background
genes drawn from the pinned scGPT vocabulary. A `t-like` cell places the T programme high and the B
programme low; a `b-like` cell does the reverse. It is built so that the two trivial handles carry
no signal: every cell is scaled to the same library size (integer rounding leaves a spread of about
0.1 %) and every cell expresses the same 348 genes, so neither total counts nor the number of
detected genes can separate the classes. What differs is *where the two programmes sit in each
cell's expression ranking*, which is what scGPT's per-cell quantile binning exposes to the model.
This is sanity evidence for the adaptation contract, not biology (NOTEBOOK_SPEC 2.0 DAT8): the
profiles are a generator rule with marker genes in it, not measured cells, and the background is
random rather than a real transcriptome.
"""

from __future__ import annotations

import csv
import hashlib
import json
import random
import re
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_GENES_PER_CELL, MIN_DETECTED_GENES, GeneVocabulary` removed — names are kernel globals defined by the carried modules

DATASET_REPRESENTATION = "io.github.kurtvalcorza.dataset.single-cell.binned-expression-labels.v1"
SAMPLE_CLASSES: tuple[str, ...] = ("b-like", "t-like")
SAMPLE_SEED = 20260918
SAMPLE_SIZE = 64  # 32 per class
PROGRAMME_GENES = 24  # per programme
BACKGROUND_GENES = 300
# Canonical PBMC lineage markers (HGNC symbols). These are what the two classes are made of; the
# model has seen their co-expression in 33 million cells, which is the point of using real ones.
T_CELL_PROGRAMME: tuple[str, ...] = (
    "CD3D",
    "CD3E",
    "CD3G",
    "CD2",
    "IL7R",
    "LTB",
    "CD7",
    "TRAC",
    "CD247",
    "LCK",
    "CD5",
    "CD6",
    "TCF7",
    "LEF1",
    "CCR7",
    "IL32",
    "GZMK",
    "CD8A",
    "CD8B",
    "CD4",
    "CD27",
    "CD28",
    "ITK",
    "ZAP70",
)
B_CELL_PROGRAMME: tuple[str, ...] = (
    "CD79A",
    "CD79B",
    "MS4A1",
    "CD19",
    "CD22",
    "BANK1",
    "PAX5",
    "VPREB3",
    "TCL1A",
    "IGHM",
    "IGHD",
    "IGKC",
    "CD74",
    "HLA-DRA",
    "HLA-DRB1",
    "HLA-DQA1",
    "HLA-DQB1",
    "HLA-DPA1",
    "HLA-DPB1",
    "FCER2",
    "CR2",
    "BLK",
    "FCRL1",
    "TNFRSF13C",
)
LIBRARY_SIZE = 20_000  # every cell is scaled to this total, so library size cannot separate the classes
HIGH_FACTOR = 6.0  # programme genes well above the background in the class that expresses them
LOW_FACTOR = 0.2  # the same genes well below it in the other class
MIN_RECORDS = 8
MAX_RECORDS = 2_000
MAX_CLASSES = 20
MIN_RECORDS_PER_CLASS = 3
MAX_ID_CHARS = 64
MAX_LABEL_CHARS = 64
REQUIRED_COLUMNS = ("id", "counts", "label")
# Canonical-looking HGNC symbols only (no clone-derived names such as RP11-22E12.2 or CTB-53D8.3),
# so the sample reads like a real expression matrix.
_CANONICAL_SYMBOL = re.compile(r"^[A-Z][A-Z0-9]{1,9}$")


def select_sample_genes(vocabulary: GeneVocabulary, seed: int = SAMPLE_SEED) -> dict[str, list[str]]:
    """The two marker programmes (checked against the vocabulary) plus a seeded background set."""
    missing = [g for g in (*T_CELL_PROGRAMME, *B_CELL_PROGRAMME) if vocabulary.resolve(g) is None]
    if missing:
        raise RuntimeError(f"marker genes absent from the pinned vocabulary: {missing}")
    rng = random.Random(seed)
    excluded = set(T_CELL_PROGRAMME) | set(B_CELL_PROGRAMME)
    candidates = sorted(
        s for s in vocabulary.gene_symbols if _CANONICAL_SYMBOL.match(s) and s not in excluded
    )
    if len(candidates) < BACKGROUND_GENES:
        raise RuntimeError(
            f"the pinned vocabulary holds {len(candidates)} canonical symbols; {BACKGROUND_GENES} needed"
        )
    return {
        "t_cell": list(T_CELL_PROGRAMME),
        "b_cell": list(B_CELL_PROGRAMME),
        "background": sorted(rng.sample(candidates, BACKGROUND_GENES)),
    }


def generate_sample_dataset(
    vocabulary: GeneVocabulary, seed: int = SAMPLE_SEED, size: int = SAMPLE_SIZE
) -> list[dict[str, Any]]:
    """`size` labelled cells (half `b-like`, half `t-like`), deterministic for a seed.

    Every cell expresses the same 348 genes and is scaled to `LIBRARY_SIZE` total counts before
    rounding. Background genes get a log-normal weight; the cell's own lineage programme is placed
    well above the background and the other lineage's well below it, so the classes differ in
    rank order — which is what per-cell quantile binning reads — rather than in library size or
    gene detection.
    """
    if size < 2 or size % 2:
        raise ValueError("size must be an even number >= 2 (one cell per class per pair)")
    genes = select_sample_genes(vocabulary, seed)
    rng = random.Random(seed + 1)
    records: list[dict[str, Any]] = []
    for i in range(size // 2):
        for label in SAMPLE_CLASSES:
            high = genes["b_cell"] if label == "b-like" else genes["t_cell"]
            low = genes["t_cell"] if label == "b-like" else genes["b_cell"]
            weights: dict[str, float] = {}
            for symbol in genes["background"]:
                weights[symbol] = rng.lognormvariate(0.0, 1.0)
            for symbol in high:
                weights[symbol] = HIGH_FACTOR * rng.lognormvariate(0.0, 0.3)
            for symbol in low:
                weights[symbol] = LOW_FACTOR * rng.lognormvariate(0.0, 0.3)
            scale = LIBRARY_SIZE / sum(weights.values())
            counts = {symbol: max(1, round(value * scale)) for symbol, value in weights.items()}
            records.append(
                {
                    "id": f"{label[0]}-cell-{i:03d}",
                    "counts": counts,
                    "label": label,
                }
            )
    return records


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """SHA-256 over the canonical (id, sorted counts, label) rows; recorded in provenance (OUT9)."""
    canon = json.dumps(
        [[r["id"], sorted(r["counts"].items()), r["label"]] for r in records], separators=(",", ":")
    )
    return hashlib.sha256(canon.encode("utf-8")).hexdigest()


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    vocabulary: GeneVocabulary | None = None,
    classes: Sequence[str] | None = None,
    min_records: int = MIN_RECORDS,
    min_per_class: int = MIN_RECORDS_PER_CLASS,
) -> dict[str, Any]:
    """Check a labelled cell dataset against the contract; return its manifest.

    Every error names the record and the violated rule (VAL4/DAT19). When `vocabulary` is supplied
    the manifest also reports how many genes per cell resolve to an scGPT token, which is the
    number that decides whether a cell can be encoded at all.
    """
    if isinstance(records, str | bytes | Mapping) or not isinstance(records, Sequence):
        raise TypeError("records must be a list of {'id', 'counts', 'label'} mappings")
    if len(records) < min_records:
        raise ValueError(f"dataset has {len(records)} records; at least {min_records} are required")
    if len(records) > MAX_RECORDS:
        raise ValueError(f"dataset has {len(records)} records; ceiling is {MAX_RECORDS}")
    seen_ids: set[str] = set()
    counts_by_class: dict[str, int] = {}
    detected: list[int] = []
    encodable: list[int] = []
    libraries: list[float] = []
    for i, rec in enumerate(records):
        if not isinstance(rec, Mapping):
            raise TypeError(f"record[{i}] must be a mapping, got {type(rec).__name__}")
        missing = [c for c in REQUIRED_COLUMNS if c not in rec]
        if missing:
            raise ValueError(
                f"record[{i}] is missing required column(s) {missing}; required: {list(REQUIRED_COLUMNS)}"
            )
        rid = str(rec["id"]).strip()
        if not rid or len(rid) > MAX_ID_CHARS:
            raise ValueError(f"record[{i}] id must be 1..{MAX_ID_CHARS} characters")
        if rid in seen_ids:
            raise ValueError(f"record[{i}] duplicates id {rid!r}")
        seen_ids.add(rid)
        cell = rec["counts"]
        if not isinstance(cell, Mapping) or not cell:
            raise ValueError(f"record[{i}] ({rid}) counts must be a non-empty mapping of gene to count")
        n_detected = 0
        total = 0.0
        for gene, value in cell.items():
            if not isinstance(gene, str) or not gene.strip():
                raise TypeError(f"record[{i}] ({rid}) has a non-string gene key {gene!r}")
            if isinstance(value, bool) or not isinstance(value, int | float):
                raise TypeError(f"record[{i}] ({rid}) count for {gene!r} must be a number")
            if value < 0 or value != value or value in (float("inf"), float("-inf")):
                raise ValueError(f"record[{i}] ({rid}) count for {gene!r} must be finite and non-negative")
            if value > 0:
                n_detected += 1
                total += float(value)
        if n_detected < MIN_DETECTED_GENES:
            raise ValueError(
                f"record[{i}] ({rid}) has {n_detected} detected gene(s); at least {MIN_DETECTED_GENES} "
                "are required to bin an expression profile"
            )
        detected.append(n_detected)
        libraries.append(total)
        if vocabulary is not None:
            n_encodable = sum(
                1 for gene, value in cell.items() if value > 0 and vocabulary.resolve(gene) is not None
            )
            if n_encodable < MIN_DETECTED_GENES:
                raise ValueError(
                    f"record[{i}] ({rid}) has {n_encodable} gene(s) in the scGPT vocabulary; at least "
                    f"{MIN_DETECTED_GENES} are required (are these human gene symbols?)"
                )
            encodable.append(n_encodable)
        label = rec["label"]
        if not isinstance(label, str) or not label.strip() or len(label) > MAX_LABEL_CHARS:
            raise ValueError(
                f"record[{i}] ({rid}) label must be a non-empty string of at most {MAX_LABEL_CHARS} chars"
            )
        counts_by_class[label] = counts_by_class.get(label, 0) + 1
    if classes is None:
        class_list = sorted(counts_by_class)
    else:
        class_list = [str(c) for c in classes]
        unknown = sorted(set(counts_by_class) - set(class_list))
        if unknown:
            raise ValueError(f"labels {unknown} are not in the class list {class_list}")
    if len(class_list) < 2:
        raise ValueError(f"classification needs at least 2 classes, found {class_list}")
    if len(class_list) > MAX_CLASSES:
        raise ValueError(f"{len(class_list)} classes exceeds the ceiling of {MAX_CLASSES}")
    thin = [c for c in class_list if counts_by_class.get(c, 0) < min_per_class]
    if thin:
        raise ValueError(f"classes {thin} have fewer than {min_per_class} records each (class coverage rule)")
    manifest: dict[str, Any] = {
        "verdict": "accepted",
        "representation": DATASET_REPRESENTATION,
        "n_records": len(records),
        "classes": class_list,
        "class_counts": {c: counts_by_class.get(c, 0) for c in class_list},
        "detected_genes": {
            "min": min(detected),
            "max": max(detected),
            "mean": round(sum(detected) / len(detected), 1),
        },
        "library_size": {
            "min": min(libraries),
            "max": max(libraries),
            "mean": round(sum(libraries) / len(libraries), 1),
        },
        "ceilings": {
            "max_genes_per_cell": MAX_GENES_PER_CELL,
            "min_detected_genes": MIN_DETECTED_GENES,
            "max_records": MAX_RECORDS,
            "max_classes": MAX_CLASSES,
            "min_records": min_records,
            "min_records_per_class": min_per_class,
        },
        "digest": dataset_digest(records),
        "findings": [],
    }
    if encodable:
        manifest["encodable_genes"] = {
            "min": min(encodable),
            "max": max(encodable),
            "mean": round(sum(encodable) / len(encodable), 1),
        }
    return manifest


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.25,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Stratified random train/validation/test split (assumes independent cells, SPL3).

    Real single-cell data is rarely independent — cells from one donor, plate or batch belong
    together — so a deployment must split by that grouping instead. The tutorial's cells are
    generated independently, which is why a random split is honest here.
    """
    if not (0.0 < val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("val_fraction and test_fraction must be in (0, 1) and sum to less than 1")
    manifest = validate_dataset(records)
    rng = random.Random(seed)
    by_class: dict[str, list[dict[str, Any]]] = {c: [] for c in manifest["classes"]}
    for rec in records:
        by_class[rec["label"]].append(dict(rec))
    out: dict[str, list[dict[str, Any]]] = {"train": [], "validation": [], "test": []}
    for cls in manifest["classes"]:
        rows = by_class[cls]
        rng.shuffle(rows)
        n_val = max(1, round(len(rows) * val_fraction))
        n_test = max(1, round(len(rows) * test_fraction))
        if len(rows) - n_val - n_test < 1:
            raise ValueError(f"class {cls!r} has {len(rows)} records; too few to leave one per split")
        out["validation"].extend(rows[:n_val])
        out["test"].extend(rows[n_val : n_val + n_test])
        out["train"].extend(rows[n_val + n_test :])
    for part in out.values():
        rng.shuffle(part)
    return out


def load_byod_dataset(source: str | Path) -> list[dict[str, Any]]:
    """Read a user-supplied cell dataset (JSON array, JSONL, or a genes-as-columns CSV).

    CSV shape: first column `id`, last column `label`, every other column a gene symbol whose cells
    hold counts; zero counts are dropped (they are never tokenised). JSON/JSONL records are
    `{"id": ..., "counts": {gene: count, ...}, "label": ...}`. Nothing is renamed or rescaled
    (VAL7); the records are then validated with `validate_dataset`. An AnnData `.h5ad` file is not
    read here — export its `X` with `var_names` as columns to CSV first.
    """
    path = Path(source)
    if not path.is_file():
        raise FileNotFoundError(f"BYOD dataset file not found: {path}")
    text = path.read_text(encoding="utf-8-sig")
    if not text.strip():
        raise ValueError(f"BYOD dataset file is empty: {path}")
    suffix = path.suffix.lower()
    records: list[dict[str, Any]] = []
    if suffix == ".csv":
        reader = csv.reader(text.splitlines())
        header = [h.strip() for h in next(reader, [])]
        if len(header) < 3 or header[0] != "id" or header[-1] != "label":
            raise ValueError(
                "CSV must start with an 'id' column, end with a 'label' column, and carry one gene per "
                f"column in between; got header {header[:3]}...{header[-1:]}"
            )
        genes = header[1:-1]
        for line_no, row in enumerate(reader, start=2):
            if not row:
                continue
            if len(row) != len(header):
                raise ValueError(f"line {line_no} has {len(row)} fields, header has {len(header)}")
            counts: dict[str, float] = {}
            for gene, value in zip(genes, row[1:-1], strict=True):
                value = value.strip()
                if not value:
                    continue
                try:
                    number = float(value)
                except ValueError as exc:
                    raise ValueError(f"line {line_no}, gene {gene!r}: {value!r} is not a number") from exc
                if number > 0:
                    counts[gene] = number
            records.append({"id": row[0].strip(), "counts": counts, "label": row[-1].strip()})
    elif suffix == ".jsonl":
        for line_no, line in enumerate(text.splitlines(), start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"line {line_no} is not valid JSON: {exc}") from exc
    elif suffix == ".json":
        try:
            data = json.loads(text)
        except json.JSONDecodeError as exc:
            raise ValueError(f"file is not valid JSON: {exc}") from exc
        if not isinstance(data, list):
            raise TypeError("JSON dataset must be a top-level array of objects")
        records = data
    else:
        raise ValueError(f"unsupported BYOD file type {suffix!r}; use .csv, .json or .jsonl")
    validate_dataset(records)
    return [dict(r) for r in records]


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write records in the BYOD CSV shape (`id`, one column per gene, `label`) as a template."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    genes = sorted({gene for r in records for gene in r["counts"]})
    with open(out, "w", encoding="utf-8", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(["id", *genes, "label"])
        for r in records:
            writer.writerow([r["id"], *(r["counts"].get(gene, 0) for gene in genes), r["label"]])
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `acf749f35bf5…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `ScGPTPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "scgpt",
  "modelId": "tdc/scGPT",
  "revision": "acf749f35bf5c0b00633838f02588272ed0d9911",
  "files": [
    {
      "path": "README.md",
      "bytes": 3307,
      "sha256": "f262f85cd1816c5263885e1198e05b99613a76872f8c48d5d840d08148034cab"
    },
    {
      "path": "config.json",
      "bytes": 467,
      "sha256": "c8bf8e037352a5cc697bbf3c35cad5a15819e889de34a3810bccd58cdfb9491a"
    },
    {
      "path": "model.safetensors",
      "bytes": 203233980,
      "sha256": "cabd40e6e22514865825975940f2c61ac1395f3317bba9f773858cd72914064c"
    },
    {
      "path": "vocab.json",
      "bytes": 1317639,
      "sha256": "ee2b2c90158eedb97c2318e49abaaed0a02c6fdf7e3f7ca6a821906413c4d2a4",
      "source": "https://dataverse.harvard.edu/api/access/datafile/10809431",
      "note": "TDC `scgpt_vocab` (gene symbol -> token id, 60,697 entries); not in the Hub repository, fetched from Harvard Dataverse by persistent file id and digest-verified"
    }
  ],
  "totalBytes": 204555393
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = ScGPTPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Sample cells, validation and split

The default dataset is generated in code with a fixed seed from the pinned vocabulary: 32 `t-like` and 32 `b-like` cells, each expressing the same 348 real gene symbols — 24 T-cell markers, 24 B-cell markers, 300 background genes — and each scaled to 20,000 total counts before rounding. `validate_dataset` checks the schema, every count, class coverage and, given the vocabulary, how many genes per cell scGPT can encode, before any model runs. `split_dataset` shuffles within each class and cuts 20 % validation / 25 % test.

Look for: 64 cells, classes `['b-like', 't-like']`, 348 detected and 348 encodable genes in every cell, library sizes within a fraction of a percent of 20,000, splits 36/12/16, and a written `outputs/scgpt_single_cell_sample_dataset.csv` in the genes-as-columns shape BYOD expects. The programmes are printed so you can see they are real markers; the cells are still a generator rule, not biology.

In [ ]:
import json
import os
from pathlib import Path

USE_BYOD = False  # @param {type:"boolean"}
VAL_FRACTION = 0.2  # @param {type:"number"}
TEST_FRACTION = 0.25  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
vocabulary = pipe.vocabulary
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    data_source = 'BYOD (' + file_name + ')'
else:
    records = generate_sample_dataset(vocabulary)
    data_source = f'synthetic marker-programme dataset (seed {SAMPLE_SEED}, {SAMPLE_SIZE} cells, real gene symbols)'
    programmes = select_sample_genes(vocabulary)
    print({'t_cell_programme': programmes['t_cell']})
    print({'b_cell_programme': programmes['b_cell']})
    print({'background_genes': len(programmes['background']), 'first_five': programmes['background'][:5]})

dataset_manifest = validate_dataset(records, vocabulary=vocabulary)
CLASSES = dataset_manifest['classes']
splits = split_dataset(records, val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION, seed=SEED)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(records, 'outputs/scgpt_single_cell_sample_dataset.csv')

print({'data_source': data_source, 'n_records': dataset_manifest['n_records'], 'classes': CLASSES, 'class_counts': dataset_manifest['class_counts']})
print({'detected_genes': dataset_manifest['detected_genes'], 'encodable_genes': dataset_manifest['encodable_genes'], 'library_size': dataset_manifest['library_size']})
print({'ceilings': dataset_manifest['ceilings'], 'digest': dataset_manifest['digest'][:16] + '...'})
print({'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)})

## 5. How a cell becomes tokens

`bin_expression` is the input encoding scGPT was trained on: detected genes are normalised to 10,000 total counts, log1p-transformed and **quantile-binned within the cell** into 51 levels (bin 0 is reserved for zeros, which are never tokenised), then listed as (gene token, bin) pairs after a `<cls>` token with value 0. Because binning is rank-based, scaling a cell's counts changes nothing — the cell below is fed twice, once multiplied by 37. Sparse count data has many ties (every gene counted once shares one value); upstream spreads tied values uniformly at random across the bins they straddle, and the checkpoint was trained on that spread, so this package does the same with a seeded generator: reproducible, and distributed like training.

The ceilings follow from the checkpoint: `MAX_GENES_PER_CELL = 1535` (1,536 positions minus `<cls>`; a cell with more detected genes keeps the most expressed and reports the truncation), `MAX_CELLS_PER_CALL = 32`, and at least `MIN_DETECTED_GENES = 10`. Unknown symbols are dropped and **reported**, never mapped onto a real gene — the packager's own tokenizer maps unknowns to id 0, which is the gene A1BG. The cell shows four rejections and one truncation.

In [ ]:
example = test_records[0]
encoded = bin_expression(example['counts'], vocabulary)
print({'id': example['id'], 'label': example['label'], 'tokens': len(encoded['tokens']), 'first_token_is_cls': encoded['tokens'][0] == vocabulary.cls_id, 'n_bins': encoded['n_bins']})
print({'top_genes': encoded['gene_symbols'][:10], 'top_bins': encoded['values'][1:11]})
print({'bottom_genes': encoded['gene_symbols'][-5:], 'bottom_bins': encoded['values'][-5:]})
scaled = bin_expression({g: v * 37 for g, v in example['counts'].items()}, vocabulary)
print({'scale_invariant': scaled['values'] == encoded['values'] and scaled['tokens'] == encoded['tokens']})
assert scaled['values'] == encoded['values']

print({'max_genes_per_cell': MAX_GENES_PER_CELL, 'max_cells_per_call': MAX_CELLS_PER_CALL, 'min_detected_genes': MIN_DETECTED_GENES, 'vocab_size': VOCAB_SIZE, 'specials': {'pad': vocabulary.pad_id, 'cls': vocabulary.cls_id, 'eoc': vocabulary.eoc_id}})
print({'validation': INPUT_SCHEMA['validation']})

probes = {
    'empty cell': {},
    'two genes': {'GAPDH': 3, 'ACTB': 5},
    'negative count': {**example['counts'], 'CD3D': -1},
    'no known symbols': {f'NOTAGENE{i}': 1.0 for i in range(12)},
}
for name, cell in probes.items():
    try:
        validate_inputs([cell], vocabulary)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

wide = {symbol: 1.0 + k for k, symbol in enumerate(vocabulary.gene_symbols[:MAX_GENES_PER_CELL + 50])}
wide_manifest = validate_inputs([wide], vocabulary)
print({'truncation_probe': wide_manifest['inputs'][0]})

input_manifest = validate_inputs([r['counts'] for r in test_records[:4]], vocabulary, names=[r['id'] for r in test_records[:4]])
print({'verdict': input_manifest['verdict'], 'n_cells': input_manifest['n_cells'], 'max_tokens_observed': input_manifest['max_tokens_observed'], 'requires_remote_code': input_manifest['requires_remote_code']})

## 6. Cell embeddings (representation, not prediction)

`pipe.embed` runs the verified encoder and returns the `<cls>` output per cell: 512 numbers. Embeddings are representations — they carry no label and no metric of their own; a downstream labelled task is what gives them meaning (EVAL9). The cell embeds eight validation cells, writes them to `outputs/scgpt_single_cell_embeddings.csv` (OUT4), and checks three properties: the same batch twice gives identical vectors; **shuffling the order of a cell's genes changes nothing** — tokens are ranked before encoding and scGPT has no positional encoding, so the assertion is exact; and embedding one cell alone versus inside a batch differs only by float accumulation order (bounded below at 1e-4, observed around 1e-6).

It also prints the geometry a reader should know about: raw `<cls>` vectors of unrelated cells have cosine similarity around 0.95 (a large shared component), so within- and between-class cosines are shown after centring on the batch mean. That centring is what Section 8's zero-training baseline relies on.

In [ ]:
import csv
import math
import random

embed_records = val_records[:8]
embedding_result = pipe.embed([r['counts'] for r in embed_records], names=[r['id'] for r in embed_records])
vectors = embedding_result['embeddings']
print({'n_cells': embedding_result['n_cells'], 'dimension': embedding_result['dimension'], 'tokens': embedding_result['tokens'], 'pooling': embedding_result['pooling']})

repeat = pipe.embed([r['counts'] for r in embed_records], names=[r['id'] for r in embed_records])['embeddings']
single = pipe.embed([embed_records[0]['counts']])['embeddings'][0]
shuffled = list(embed_records[0]['counts'].items())
random.Random(3).shuffle(shuffled)
reordered = pipe.embed([dict(shuffled)])['embeddings'][0]
order_diff = max(abs(a - b) for a, b in zip(reordered, single))
cross_batch = max(abs(a - b) for a, b in zip(single, vectors[0]))
print({'same_batch_twice_identical': repeat == vectors, 'gene_order_max_abs_diff': order_diff, 'single_vs_batch_max_abs_diff': cross_batch, 'note': 'gene order in the input never matters (tokens are ranked before encoding, and the model has no positional encoding); batch composition changes float accumulation order at the 1e-6 level'})
assert repeat == vectors
assert order_diff == 0.0
assert cross_batch < 1e-4

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b)) / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

mean = [sum(v[d] for v in vectors) / len(vectors) for d in range(len(vectors[0]))]
centred = [[x - m for x, m in zip(v, mean)] for v in vectors]
raw_pairs, within, between = [], [], []
for i in range(len(embed_records)):
    for j in range(i + 1, len(embed_records)):
        raw_pairs.append(cosine(vectors[i], vectors[j]))
        (within if embed_records[i]['label'] == embed_records[j]['label'] else between).append(cosine(centred[i], centred[j]))
print({'raw_cosine_between_cells_mean': round(sum(raw_pairs) / len(raw_pairs), 4), 'centred_cosine_within_class': round(sum(within) / len(within), 4) if within else None, 'centred_cosine_between_classes': round(sum(between) / len(between), 4) if between else None, 'note': 'inspection only; embeddings are unlabelled representations'})

with open('outputs/scgpt_single_cell_embeddings.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'label', 'tokens'] + [f'dim_{k}' for k in range(embedding_result['dimension'])])
    for r, n_tokens, vec in zip(embed_records, embedding_result['tokens'], vectors):
        writer.writerow([r['id'], r['label'], n_tokens] + [f'{x:.6f}' for x in vec])
print('wrote outputs/scgpt_single_cell_embeddings.csv')

## 7. The masked-expression objective, probed honestly

scGPT was pretrained to predict the bins of masked genes from the rest of the cell, and the checkpoint carries that decoder. `pipe.predict_masked` masks a seeded 15 % of each cell's gene positions with the training-time mask value (−1) and reports, per cell, the mean absolute error in bins and the Pearson correlation between predicted and true bins. A second probe asks the question a biologist would: mask the T-cell markers in a `t-like` cell and in a `b-like` cell — does the decoder predict them higher where the rest of the cell says "T lymphocyte"?

Read the numbers as they come. On this checkpoint the decoder's predictions sit in a narrow band around bin 28–32 whatever the context, so the masked-prediction correlation is weak and the lineage probe shows no conditioning. That is a property of the shipped decoder, not of the encoder: Section 8 shows the *frozen embeddings* separate the lineages perfectly with no training at all, and the same encoder separated real B, T and monocyte cells from a public PBMC dataset at build time (recorded in the model card). A DIMER profile should therefore expose embeddings and classification, not imputation.

In [ ]:
masked = pipe.predict_masked([r['counts'] for r in val_records[:8]], names=[r['id'] for r in val_records[:8]], mask_fraction=0.15, seed=SEED)
pearsons = [c['pearson'] for c in masked['cells'] if c['pearson'] is not None]
maes = [c['mean_absolute_error_bins'] for c in masked['cells']]
print({'mask_fraction': masked['mask_fraction'], 'mask_value': masked['mask_value'], 'n_bins': masked['n_bins']})
print({'mean_pearson': round(sum(pearsons) / len(pearsons), 4), 'mean_absolute_error_bins': round(sum(maes) / len(maes), 3), 'note': masked['note']})
for c in masked['cells'][:3]:
    print(c)

if not USE_BYOD:
    lineage_probe = {}
    for label in CLASSES:
        cells_of = [r['counts'] for r in val_records if r['label'] == label][:4]
        for marker_name, markers in (('T markers', T_CELL_PROGRAMME), ('B markers', B_CELL_PROGRAMME)):
            probe = pipe.predict_masked_genes(cells_of, markers)
            lineage_probe[f'{label} / masked {marker_name}'] = {'predicted_mean_bin': probe['predicted_mean_bin'], 'true_mean_bin': probe['true_mean_bin']}
    for key, row in lineage_probe.items():
        print({key: row})
    print({'reading': 'a decoder that used lineage context would predict T markers high in t-like cells and low in b-like cells; the shipped decoder predicts a similar bin either way'})
else:
    lineage_probe = None

## 8. Baselines on the test split, including one with no training at all

Three predictors set the floor before any gradient step (EVAL10/EVAL11). `pipe.nearest_centroid_evaluate` is the frozen embedding's own separability: centre the `<cls>` vectors on the training mean, fit one centroid per class on the **training split only** (SPL8), and assign each test cell to the nearest centroid by cosine. It is the zero-training number every adapted result must be read against — if it is already perfect, fine-tuning has nothing to add on this data. `majority_baseline` predicts the most frequent training class (0.5 on a balanced split). `library_size_baseline` thresholds on total counts, fitted on the training split; the sample scales every cell to the same total, so it sits at chance by construction, which is the point.

On your own data, read the library-size baseline first: if total counts already separate your labels, the labels track sequencing depth, not biology.

In [ ]:
baseline_centroid = pipe.nearest_centroid_evaluate(train_records, test_records, classes=CLASSES)
print({k: baseline_centroid[k] for k in ('baseline', 'fitted_on', 'n', 'accuracy', 'macro_f1', 'auroc')})
baseline_majority = majority_baseline(train_records, test_records, CLASSES)
print({k: baseline_majority[k] for k in ('baseline', 'predicted_label', 'accuracy', 'macro_f1')})
baseline_library = library_size_baseline(train_records, test_records, CLASSES)
print({k: baseline_library[k] for k in ('baseline', 'rule', 'train_accuracy', 'accuracy', 'macro_f1', 'auroc')})

## 9. Bounded fine-tuning

`pipe.adapt` copies the verified encoder, puts a newly initialised linear head over its `<cls>` output, freezes every parameter except the head and the last `TRAINABLE_LAYERS` encoder layers, and runs AdamW with the hyperparameters below (FT4/FT6): tutorial values chosen for a few tens of seconds of CPU, not production settings. Validation metrics are computed after each epoch for **monitoring only**; the final epoch's weights are kept, so no selection happens on the validation split (EVAL14). Training loss going down is optimisation evidence, not task-quality evidence (FT7) — Section 10 is where quality is measured.

`TRAINABLE_LAYERS = 0` trains the head alone on `<cls>` embeddings the frozen encoder computes once — fast, and worth comparing: the raw embeddings share a large common component, so a bare linear head learns more slowly than the centred nearest-centroid rule above.

In [ ]:
import time

EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_LAYERS = 1  # @param {type:"integer"}

started = time.perf_counter()
adapt_result = pipe.adapt(
    train_records,
    val_records,
    classes=CLASSES,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    trainable_layers=TRAINABLE_LAYERS,
    seed=SEED,
)
adapt_seconds = round(time.perf_counter() - started, 2)
print({'method': adapt_result['method'], 'trainable_parameters': adapt_result['trainable_parameters'], 'total_parameters': adapt_result['total_parameters'], 'precision': adapt_result['precision'], 'device': pipe.device, 'seconds': adapt_seconds})
for step in adapt_result['history']:
    print(step)

## 10. Held-out evaluation

`pipe.evaluate` classifies every cell of a split and reports `accuracy`, `macro_f1` (the unweighted mean of per-class F1, which exposes a model that ignores a class), per-class precision/recall/F1 with support, and `auroc` (ranking quality of the positive-class score, independent of the argmax threshold). The **test split** was never used for training or monitoring, so its numbers are the independent evidence (SPL6/SPL7). These are tutorial metrics on a synthetic 16-cell split (EVAL6): one holdout, no dispersion estimate. The report, with all three baselines and the deltas against the zero-training centroid rule and the majority rule, is written to `outputs/scgpt_single_cell_evaluation_report.json`.

In [ ]:
val_metrics = pipe.evaluate(val_records)
test_metrics = pipe.evaluate(test_records)
print({'split': 'validation', **{k: val_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
print({'split': 'test', **{k: test_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
for cls_name, row in test_metrics['per_class'].items():
    print({'class': cls_name, **row})

evaluation_report = {
    'task': 'cell-state classification (bounded fine-tuning of scGPT)',
    'evidence': 'tutorial sample-sanity metrics on one stratified holdout of synthetic marker-programme cells; not a benchmark',
    'estimation': 'single train/validation/test split, seed ' + str(SEED) + ', no dispersion estimate',
    'data_source': data_source,
    'dataset_digest': dataset_manifest['digest'],
    'classes': CLASSES,
    'splits': {'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)},
    'baselines': {'nearest_centroid': baseline_centroid, 'majority': baseline_majority, 'library_size': baseline_library},
    'validation_metrics': val_metrics,
    'test_metrics': test_metrics,
    'delta_vs_nearest_centroid': {k: round(test_metrics[k] - baseline_centroid[k], 4) for k in ('accuracy', 'macro_f1')},
    'delta_vs_majority': {k: round(test_metrics[k] - baseline_majority[k], 4) for k in ('accuracy', 'macro_f1')},
    'masked_expression_probe': {'mean_pearson': round(sum(pearsons) / len(pearsons), 4), 'mean_absolute_error_bins': round(sum(maes) / len(maes), 3), 'lineage_probe': lineage_probe},
    'adaptation': {k: v for k, v in adapt_result.items() if k != 'trainable_parameter_names'},
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/scgpt_single_cell_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
print({'delta_vs_nearest_centroid': evaluation_report['delta_vs_nearest_centroid'], 'delta_vs_majority': evaluation_report['delta_vs_majority'], 'report': 'outputs/scgpt_single_cell_evaluation_report.json'})

## 11. Inference on new cells, artifact export and fresh reload

`pipe.classify` returns, per cell, the argmax `label`, its `score` and the full `scores` dictionary in class order. The scores are softmax outputs of a head trained on a few dozen cells — **not calibrated probabilities** (UNC2); the only decision rule is argmax (UNC3). The new cells are generated with a different seed so they were never seen in any split.

`pipe.save_artifact` writes the trained tensors as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the class order, the tensor names, the file size and SHA-256, and the adaptation configuration (OUT8). `ScGPTPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digests **before** deserialising, rebuilds the classifier and overlays the tensors — a fresh object from files, not the in-memory model (VER2). The cell asserts identical labels and scores within `1e-5` (VER4).

In [ ]:
if USE_BYOD:
    new_records = test_records[:6]
    new_source = 'first six BYOD test-split cells'
else:
    new_records = generate_sample_dataset(vocabulary, seed=7, size=6)
    new_source = 'freshly generated marker-programme cells (seed 7)'
inference_result = pipe.classify([r['counts'] for r in new_records], names=[r['id'] for r in new_records])
predictions = inference_result['predictions']
print({'new_source': new_source, 'decision_rule': inference_result['decision_rule']})
n_match = 0
for p, r in zip(predictions, new_records):
    n_match += p['label'] == r['label']
    print({'id': p['id'], 'predicted': p['label'], 'score': round(p['score'], 4), 'true_label': r['label']})
print({'matches': n_match, 'of': len(new_records), 'note': 'sanity check on generated labels, not an evaluation'})

with open('outputs/scgpt_single_cell_predictions.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'predicted_label', 'score'] + [f'score_{c}' for c in CLASSES])
    for p in predictions:
        writer.writerow([p['id'], p['label'], f"{p['score']:.6f}"] + [f"{p['scores'][c]:.6f}" for c in CLASSES])

artifact_dir = Path('outputs/scgpt_single_cell_adapter')
pipe.save_artifact(artifact_dir, metadata={'data_source': data_source, 'dataset_digest': dataset_manifest['digest'], 'test_metrics': {k: test_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
with open(artifact_dir / ARTIFACT_MANIFEST_NAME, encoding='utf-8') as f:
    artifact_manifest = json.load(f)
print({'format': artifact_manifest['format'], 'base_model': artifact_manifest['base_model'], 'requires_remote_code': artifact_manifest['requires_remote_code'], 'n_tensors': len(artifact_manifest['tensors']), 'files': artifact_manifest['files']})

reloaded_pipe = ScGPTPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR)
reloaded_result = reloaded_pipe.classify([r['counts'] for r in new_records], names=[r['id'] for r in new_records])
max_score_diff = 0.0
for before, after in zip(predictions, reloaded_result['predictions']):
    assert before['id'] == after['id'] and before['label'] == after['label'], f'reload parity failure on {before["id"]}'
    max_score_diff = max(max_score_diff, abs(before['score'] - after['score']))
assert max_score_diff < 1e-5, f'reload score drift {max_score_diff}'
print({'reload_parity': 'PASS', 'labels_equal': True, 'max_abs_score_diff': max_score_diff})

## 12. Result export and provenance

The last output, `outputs/scgpt_single_cell_result.json`, gathers what a reader needs to interpret the files above: the notebook source revision, the model id, immutable revision and licence, the vocabulary's source and digest, the fact that no remote code was executed, the dataset source and digest, the masked-expression probe, the adaptation configuration, baseline and held-out metrics, the new-cell predictions, the artifact manifest, the reload-parity result, and the runtime versions and device (OUT6/OUT7). No credential is involved anywhere in this notebook, so none can leak into it (OUT10).

In [ ]:
import platform

vocab_entry = next(e for e in MANIFEST['files'] if e['path'] == VOCAB_NAME)
result_payload = {
    'task': 'cell-state classification adaptation (scGPT)',
    'pipeline_class': 'ScGPTPipeline',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'remote_code_executed': False,
    'vocabulary': {'source': vocab_entry.get('source'), 'sha256': vocab_entry['sha256'], 'size': VOCAB_SIZE},
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'notebook_source': NOTEBOOK_SOURCE,
    'data_source': data_source,
    'dataset_manifest': dataset_manifest,
    'embedding_summary': {'n_cells': embedding_result['n_cells'], 'dimension': embedding_result['dimension'], 'pooling': embedding_result['pooling'], 'gene_order_max_abs_diff': order_diff, 'single_vs_batch_max_abs_diff': cross_batch},
    'evaluation_report': evaluation_report,
    'inference': {'new_source': new_source, 'decision_rule': inference_result['decision_rule'], 'predictions': predictions},
    'artifact_format': ARTIFACT_FORMAT,
    'artifact_format_version': ARTIFACT_FORMAT_VERSION,
    'artifact_manifest': artifact_manifest,
    'reload_parity': {'labels_equal': True, 'max_abs_score_diff': max_score_diff},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'safetensors': safetensors.__version__,
        'numpy': numpy.__version__,
        'device': pipe.device,
        'precision': 'float32',
    },
}
with open('outputs/scgpt_single_cell_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

The frozen encoder already separates cells whose T-cell and B-cell marker programmes are swapped — the zero-training nearest-centroid rule scores as well as the fine-tuned head — and the bounded adaptation preserves that on an independent split. That is the claim: scGPT's `<cls>` embedding carries lineage structure it learned from 33 million real cells, it reads that structure out of nothing but a binned expression ranking, and the adaptation contract works on top of it. The masked-expression decoder, by contrast, is not a usable imputer as shipped, and the notebook says so with numbers rather than skipping the question.

The test split has 16 synthetic cells, the metrics come from one seeded holdout with no dispersion estimate, and the classes are a generator rule with real marker genes in it rather than measured cells. So a perfect score says the contract works, not that scGPT annotates cell types at any published accuracy, integrates batches, or predicts perturbations — none of which this repository exercises.

Three things to carry to real data. **Symbols:** the vocabulary is HGNC symbols, case-insensitively matched; Ensembl ids are unknown and dropped, and the input manifest tells you how many genes survived — read it before you trust an embedding. **Splits:** cells from one donor, plate or batch belong together; split by that grouping, not at random. **Depth:** read the library-size baseline first — if total counts predict your labels, so will any model, for the wrong reason.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model **and a vocabulary from a second source**, rebuild the encoder without any upstream code, validate the demonstrated dataset contract, execute bounded fine-tuning, evaluate against a zero-training baseline and two trivial ones on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or biological validity.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_LAYERS = 0` to train the head alone on cached embeddings and watch it learn more slowly than the centred centroid rule; raise `TRAINABLE_LAYERS` to 12 to fine-tune the whole encoder and compare the time; or bring your own labelled cells through BYOD and read the nearest-centroid and library-size baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/scgpt-single-cell-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/scgpt-single-cell-pipeline/blob/main/MODEL_CARD.md
- Upstream model (safetensors packaging): https://huggingface.co/tdc/scGPT
- Upstream code: https://github.com/bowang-lab/scGPT
- Cui, H., Wang, C., Maan, H., Pang, K., Luo, F., Duan, N., Wang, B. (2024). scGPT: toward building a foundation model for single-cell multi-omics using generative AI. Nature Methods 21, 1470–1480. https://doi.org/10.1038/s41592-024-02201-0
- Velez-Arce, A., et al. (2024). Signals in the Cells: Multimodal and Contextualized Machine Learning Foundations for Therapeutics. NeurIPS 2024 Workshop on AI for New Drug Modalities (the TDC packaging).